## Which batsmen give best value-for-money? (Auction ROI Index)
Calculated runs per match, strike rate, consistency score.  Which players give you most bang for your auction rupee?

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

In [26]:
# Apply cleaning
matches['city'] = matches['city'].fillna('Unknown')
matches['player_of_match'] = matches['player_of_match'].fillna('Unknown')
matches['winner'] = matches['winner'].fillna('No Result')
matches['season'] = matches['season'].replace({
    '2007/08': '2008', '2009/10': '2010', '2020/21': '2020'
}).astype(int)

name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiant':'Rising Pune Supergiants'
}
for col in ['team1','team2','winner','toss_winner']:
    matches[col] = matches[col].replace(name_map)

deliveries = deliveries[deliveries['inning'] <= 2]  # remove super overs

In [27]:
# ── Q1 — Batsman Value (03_analysis_batting.ipynb) ──
batsman_stats = deliveries.groupby('batter').agg(      # note: 'batter' not 'batsman'
    total_runs=('batsman_runs', 'sum'),
    balls_faced=('batsman_runs', 'count'),
    innings_played=('match_id', 'nunique')
).reset_index()

batsman_stats['strike_rate'] = (batsman_stats['total_runs'] / batsman_stats['balls_faced']) * 100
batsman_stats['avg_per_innings'] = batsman_stats['total_runs'] / batsman_stats['innings_played']

# Only players with 30+ innings — removes noise
qualified = batsman_stats[batsman_stats['innings_played'] >= 30].copy()
qualified = qualified.sort_values('avg_per_innings', ascending=False)
print(qualified.head(15))

           batter  total_runs  balls_faced  innings_played  strike_rate  \
289      KL Rahul        4683         3572             122   131.103024   
473    RD Gaikwad        2380         1781              65   133.632791   
542      SE Marsh        2477         1908              69   129.821803   
147     DA Warner        6565         4844             184   135.528489   
124      CH Gayle        4965         3502             141   141.776128   
365     ML Hayden        1107          838              32   132.100239   
352    MEK Hussey        1977         1648              58   119.963592   
242    JC Buttler        3582         2517             106   142.312277   
188  F du Plessis        4571         3435             138   133.071325   
631       V Kohli        8004         6232             244   128.433890   
592  Shubman Gill        3216         2432              99   132.236842   
256   JM Bairstow        1589         1132              50   140.371025   
122       CA Lynn        

In [28]:
batsman_stats.head()

,batter,total_runs,balls_faced,innings_played,strike_rate,avg_per_innings
0,A Ashish Reddy,280,196,23,142.857143,12.173913
1,A Badoni,634,505,35,125.544554,18.114286
2,A Chandila,4,7,2,57.142857,2.000000
3,A Chopra,53,75,6,70.666667,8.833333
4,A Choudhary,25,20,3,125.000000,8.333333


In [29]:
qualified.head()

,batter,total_runs,balls_faced,innings_played,strike_rate,avg_per_innings
289,KL Rahul,4683,3572,122,131.103024,38.385246
473,RD Gaikwad,2380,1781,65,133.632791,36.615385
542,SE Marsh,2477,1908,69,129.821803,35.898551
147,DA Warner,6565,4844,184,135.528489,35.679348
124,CH Gayle,4965,3502,141,141.776128,35.212766


Batsman Value

KL Rahul leads all IPL batsmen with 38.4 runs per innings across 122 innings — the most consistent top-order performer in the tournament's history
H Klaasen is the most explosive value pick — 162 strike rate with a 31+ average in just 32 innings, meaning he scores fast without throwing his wicket away
Virat Kohli's 244 innings is the most of anyone in the top 15, proving elite consistency isn't a fluke — it scales across 17 seasons of the toughest T20 cricket